# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [12]:
%load_ext autoreload
%autoreload 2

import json
from _campaign_lib import *

# --- Services ---
svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
campaign_config = {
    "sample_size": 15,              # queries per eval step (service default: all)
    "exploration_sample_size": 10,  # queries per scan/grid point (can be smaller)
    "eval_llm": {
        "model":       "moonshotai/kimi-k2-instruct-0905",
        "provider":    "groq",
        "temperature": 0.4,
    },
    "pipeline_params": None,        # set by configure_pipeline()
}

print(f"  Interrupt of cells can take up to 60 seconds!")
print(f"  If a dialog pops up, click 'Cancel' and wait 20 seconds.")

2026-03-23 14:24:22 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Backend: http://127.0.0.1:8000


2026-03-23 14:24:22 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Pipeline: termnorm (6 steps)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md
  Interrupt of cells can take up to 60 seconds!
  If a dialog pops up, click 'Cancel' and wait 20 seconds.


In [13]:
#@title Task context decomposition
task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, svc)

2026-03-23 14:24:23 WARNING  [langfuse] Prompt 'optimizer_restructure-label:production' not found during refresh, evicting from cache.


TASK CONTEXT DECOMPOSITION
  domain: Life Cycle Assessment
  pipeline_purpose: Normalize unstructured material descriptions to standardized LCA database entries for environmental impact calculations.
  data_characteristics: Mixed-language industrial shorthand (alloy codes, brand names, standards) with geographic tags, requiring domain knowledge to decode.
  optimization_goals: Maximize exact matches to database entries while correctly identifying no-match cases; handle brand name decoding and geographic variant selection.
  key_challenges: Brand name decoding (Makrolon=polycarbonate), indirect standard references (ISO 898-1=carbon steel), geographic variants, evolving database naming, no-match identification.


In [14]:
#@title Load datasets & session terms
train_data, session_terms = prepare_datasets(
    svc["store"], svc.get("backend_id", ""),
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
svc["session_terms"] = session_terms


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [15]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline_ps, eval_data, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline_ps, eval_data, campaign_config, svc,
    )


BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     111
  Match Database Aliases         716
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [16]:
#@title Experiment dashboard
# Set to a short hex ID (e.g. '68e2c5') to resume a specific experiment.
# The system adds prefixes (cycle_, scan_, etc.) per data type.
# Set to None to auto-detect from current campaign_config + eval_data.
EXPERIMENT_ID = '68e2c53845c3' #None
EXPERIMENT_ID = '77e7e77777e7' #None


# When EXPERIMENT_ID is set, load stored config → overrides notebook variables
if EXPERIMENT_ID:
    stored_cfg = load_experiment_config(svc["store"], svc["backend_id"], EXPERIMENT_ID)
    if stored_cfg:
        pp_override = apply_experiment_overrides(campaign_config, stored_cfg)
        if pp_override:
            pipeline_params = pp_override
        print(f"  Loaded config from experiment {EXPERIMENT_ID}")

show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    pipeline_params=locals().get("pipeline_params"),
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)

2026-03-23 14:24:25 WARNING  [api.services.campaign.campaign_init] No campaign matching '77e7e77777e7'
2026-03-23 14:24:25 WARNING  [api.services.campaign.campaign_init] No campaign matching '77e7e77777e7'


## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

In [17]:
#@title 3a. Smart Search — Browse variant library
# display_variant_library()
# Filter examples:
# display_variant_library(source="PromptWizard")
# display_variant_library(axes=["thinking_style", "persona"])

In [18]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)

You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  },
  {
    "name": "llm_ranking",
    "node_role": "ranker"
  }
]

## Task Context
- **domain**: Life Cycle Assessment
- **pipeline_purpose**: Normalize unstructured material descriptions to standa

In [19]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=locals().get("task_context") or locals().get("TASK_DESCRIPTION", ""),
)

SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Task context: Life Cycle Assessment — Normalize unstructured material descriptions to standardized
  Calling moonshotai/kimi-k2-instruct-0905 ...

----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Shapes search space for brand decoding
     Values: ['LCA ', 'material database ', 'trade name ']
  2. [HIGH] query_suffix (pipeline_param) -- step: web_search
     Captures geographic variants
     Values: [' ecoinvent', ' GaBi', ' {GLO}']
  3. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Controls recall for shorthand codes
     Values: [657585]
  4. [MEDIUM] max_token_candidates (pipeline

In [20]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 10  # queries per scan variant (0 = use all)

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'query_prefix': ['what material is', 'identify LCA database name for', 'translate trade name'],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    # 'profiling_max_tokens': [512, 1024, 2048], # -> Going to cause lots of Errors.
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['what material is', 'identify LCA database name for', 'translate trade name']
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material 

In [21]:
#@title Prepare scan baseline
# Scan always uses fresh pipeline defaults (not experiment overrides) so the
# baseline content hash matches previous runs regardless of EXPERIMENT_ID.
scan_pipeline_params = configure_pipeline(svc, campaign_config)
scan_baseline_sp, scan_coverage = await prepare_scan_baseline(
    baseline_ps, campaign_config,
    pipeline_params=scan_pipeline_params,
    svc=svc, scan_variants=scan_variants,
)

2026-03-23 14:24:28 INFO     [api.services.search.context] restructure_context_cached: hit (alias group)
2026-03-23 14:24:28 INFO     [api.services.search.coverage] build_prompt_result_index: 1 runs -> 1 unique prompts, 10 total query results


Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Rank 20 candidate strings by how well they match a target entity profile and cor...
    problem_description: Given an entity profile (JSON) and a core concept, score and rank 20 candidate s...
    instruction: TASK 1: Analyze profile and core concept (PRIMARY FACTOR - 70% WEIGHT)
- Summari...
    thinking_style: Think step-by-step
    answer_format: Valid JSON with reasoning string and ranked_candidates array of 20 objects each ...
  Search baseline: 69bceaf7fabe (render: 2002 chars)

  Historical data: 10 results across 1 unique prompts
  Matching runs (sp_hash): 0, 0 cached results

  Scan variant coverage (0 matching runs):
    max_token_candidates     (not tested)
    query_prefix             (not tested)
    profiling_schema         (not tested)
    profiling

In [22]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    scan_baseline_sp, scan_variants, eval_data,
    sample_size=locals().get('scan_sample_size', 10),
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

Running sensitivity scan...

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Rank 20 candidate strings by how well they match a target entity profile and cor...
    problem_description: Given an entity profile (JSON) and a core concept, score and rank 20 candidate s...
    instruction: TASK 1: Analyze profile and core concept (PRIMARY FACTOR - 70% WEIGHT)
- Summari...
    thinking_style: Think step-by-step
    answer_format: Valid JSON with reasoning string and ranked_candidates array of 20 objects each ...

  Axes: 5, variants: 17, queries/variant: 10, cached results: 10
  Estimated calls: ~170
  Evaluating baseline...
        HIT     [llm]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 19.4s
                  ⚠ web_search: 7 of 15 fetched URLs returned content (8 filtered: 6×skip_extension, 1×too_short, 1×http_403)

  [INTERRUPTED] Sensitivity scan
  Saved: completed evaluations saved via evaluate_p

In [ ]:
# #@title Scan analytics (uncomment to display)
# if scan_df is not None and not scan_df.empty:
#     show_scan_leaderboard(scan_df, axis_profiles)
#     difficulty_df = show_scan_query_difficulty(svc["store"], svc["backend_id"])

In [ ]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

No scan data available. Run sensitivity scan first.
Updated pipeline_params: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']}


TypeError: 'NoneType' object is not subscriptable

### 3b. Grid Search

<details>
<summary>Grid search cells (click to expand)</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing. All cells below are commented out by default.

**To activate:** uncomment cells below and run in order. Grid search evaluates all combinations of prompt fields and pipeline params — expect 100–500+ backend calls depending on `grid_budget` and `sample_size` in `campaign_config["grid_search"]`.

</details>

In [ ]:
# #@title Grid campaign overview (existing plans)
# merge_plans = False  # Set True to combine results from multiple plans
# grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
# merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
# #@title Build or resume grid plan
# gs = campaign_config["grid_search"]

# llm_client, llm_model = setup_llm(campaign_config)

# (
#     grid_plan_id, grid_points, grid_state_lookup,
#     grid_axes, layer1_fields, grid_baseline,
# ) = await resume_or_build_grid(
#     campaign_config, baseline, llm_client, llm_model,
#     svc["store"], svc["backend_id"],
#     improvement_areas=campaign_config.get("improvement_areas", ""),
# )

# print(f"Grid points: {len(grid_points)}")
# print(f"Plan ID: {grid_plan_id}")

In [ ]:
# #@title Run grid search
# grid_df = await run_grid_search(
#     grid_points, grid_state_lookup, eval_data,
#     campaign_config["eval_llm"],
#     plan_id=grid_plan_id,
#     svc=svc,
#     pipeline_params=campaign_config.get("pipeline_params"),
#     sample_size=gs.get("sample_size", 1),
#     shared_queries=gs.get("shared_queries", False),
#     grid_seed=gs.get("seed", 42),
# )

In [ ]:
# #@title Display grid results
# _display_df = merged_grid_df if merged_grid_df is not None else grid_df
# display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
# #@title LLM analysis of grid results
# _analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
# llm_client, llm_model = setup_llm(campaign_config)
# grid_analysis = await analyze_grid_results(
#     _analysis_df, grid_axes, llm_client, model=llm_model,
# )

In [ ]:
# #@title Select grid winner and seed campaign
# grid_winner = select_and_seed_grid_winner(
#     grid_df, merged_grid_df, grid_state_lookup,
#     grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
# )

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=locals().get("scan_df"),
    axis_profiles=locals().get("axis_profiles"),
    scan_variants=locals().get("scan_variants"),
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 10.0%
  Baseline prompt        : TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
  ------------------------------------------------------------------
  Max rounds             : 3
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=None
  L3 (modify plan)       : enabled, patience=None
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE IN

In [ ]:
#@title Run optimization (feedback cycle)
# Force-reload api modules (ensures code edits take effect without kernel restart)
import importlib, sys
for _m in [
    "api.services.campaign.escalation",
    "api.services.campaign.layer_transitions",
    "api.services.campaign.critique",
    "api.services.campaign.models",
    "api.services.prompt_optimizer",
    "api.services.campaign.feedback_cycle",
]:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc,
    pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=locals().get("EXPERIMENT_ID"),
    task_context=locals().get("task_context"),
)

In [ ]:
#@title 5. Results — Campaign comparison, flip tracking, lineage
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=locals().get("EXPERIMENT_ID"),
)

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)